## Step 3 — ResNet Model Setup

## Import PyTorch

In [32]:
import torch
import torch.nn as nn

print("PyTorch version:", torch.__version__)

PyTorch version: 2.13.0+cpu


In [34]:
## Create the residual block

In [36]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()

        self.conv1 = nn.Conv2d(
            channels,
            channels,
            kernel_size=3,
            padding=1,
            bias=False
        )

        self.bn1 = nn.BatchNorm2d(channels)

        self.relu = nn.ReLU(inplace=True)

        self.conv2 = nn.Conv2d(
            channels,
            channels,
            kernel_size=3,
            padding=1,
            bias=False
        )

        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out = out + identity
        out = self.relu(out)

        return out

In [38]:
## Create the 10-block ResNet

In [40]:
class ResNetCIFAR(nn.Module):
    def __init__(self, num_blocks=10, channels=128, num_classes=10):
        super().__init__()

        self.channels = channels

        self.stem = nn.Sequential(
            nn.Conv2d(
                3,
                channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True)
        )

        self.blocks = nn.ModuleList([
            ResidualBlock(channels)
            for _ in range(num_blocks)
        ])

        self.global_average_pool = nn.AdaptiveAvgPool2d((1, 1))

        self.classifier = nn.Linear(
            channels,
            num_classes
        )

    def forward(self, x):
        x = self.stem(x)

        for block in self.blocks:
            x = block(x)

        x = self.global_average_pool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)

        return x

    def get_block_representations(self, x):
        x = self.stem(x)

        representations = []

        for block in self.blocks:
            x = block(x)
            representations.append(x)

        return representations

In [42]:
## Create the model

In [44]:
model = ResNetCIFAR(
    num_blocks=10,
    channels=128,
    num_classes=10
)

In [46]:
## Check the number of parameters

In [48]:
total_parameters = sum(
    p.numel()
    for p in model.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Total parameters:", total_parameters)
print("Trainable parameters:", trainable_parameters)

Total parameters: 2959242
Trainable parameters: 2959242


In [22]:
## Check the forward pass

In [30]:
dummy_input = torch.randn(4, 3, 32, 32)

with torch.no_grad():
    representations = model.get_block_representations(dummy_input)

print("Number of representations:", len(representations))

for i, representation in enumerate(representations):
    print(
        f"Block {i + 1}:",
        representation.shape
    )

Number of representations: 10
Block 1: torch.Size([4, 128, 32, 32])
Block 2: torch.Size([4, 128, 32, 32])
Block 3: torch.Size([4, 128, 32, 32])
Block 4: torch.Size([4, 128, 32, 32])
Block 5: torch.Size([4, 128, 32, 32])
Block 6: torch.Size([4, 128, 32, 32])
Block 7: torch.Size([4, 128, 32, 32])
Block 8: torch.Size([4, 128, 32, 32])
Block 9: torch.Size([4, 128, 32, 32])
Block 10: torch.Size([4, 128, 32, 32])
